# Module 9 lab · Language models from scratch

The code from every lesson of [this module](https://aicourses.dev/research/language-models), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 9.2 Text as probability: a bigram language model

The smallest model that writes text counts which letter follows which, a pair called a bigram, and turns the counts into probabilities. You score it by how surprised it is by real text: the same cross-entropy loss, here called negative log-likelihood, and a friendlier form called perplexity.

[Read the lesson](https://aicourses.dev/research/language-models/bigram-language-model)

### Writing new names

First, get the names and keep 10% aside as a validation set, as in Module 8. The model never counts them; you score it on them.

In [ ]:
import random
import urllib.request

url = ("https://raw.githubusercontent.com/karpathy/"
       "makemore/master/names.txt")
urllib.request.urlretrieve(url, "names.txt")
words = open("names.txt").read().splitlines()
print(len(words), words[:4])
# 32033 ['emma', 'olivia', 'ava', 'isabella']

random.seed(42)              # same shuffle every run
random.shuffle(words)
n = int(0.9 * len(words))
train, val = words[:n], words[n:]
print(len(train), len(val))   # 28829 3204

The list is the 32,033 most common names from the US Social Security Administration’s 2018 baby-name data, collected by Andrej Karpathy for his makemore project. Now count the pairs. `stoi` (“string to index”) gives each symbol its row number, and `zip(s, s[1:])` walks along the name one neighbouring pair at a time:

In [ ]:
import torch

chars = ".abcdefghijklmnopqrstuvwxyz"   # "." = start or end
stoi = {ch: i for i, ch in enumerate(chars)}

N = torch.zeros(27, 27)       # row: this letter, column: next
for w in train:
    s = "." + w + "."
    for a, b in zip(s, s[1:]):
        N[stoi[a], stoi[b]] += 1

q = stoi["q"]
print(N[q, stoi["u"]].item(), N[q].sum().item())   # 185.0 242.0
print((N == 0).sum().item())  # 104 pairs never seen

Then smooth, divide each row by its total, and sample:

In [ ]:
P = (N + 1) / (N + 1).sum(dim=1, keepdim=True)
print(round(P[q, stoi["u"]].item(), 3))   # 0.691

g = torch.Generator().manual_seed(42)
for _ in range(5):
    i, name = 0, ""                     # start at "."
    while True:
        i = torch.multinomial(P[i], 1, generator=g).item()
        if i == 0:                      # back to ".": the end
            break
        name += chars[i]
    print(name)

### Perplexity: the friendlier number

An NLL like emma’s 2.51 is hard to picture. **Perplexity** turns it into something you can imagine: $e^{\text{NLL}}$, the number of equally likely choices that would leave you just as unsure. The know-nothing model has a perplexity of $e^{\ln 27} = 27$. A perplexity of 4 means as unsure as picking among 4 letters.

In [ ]:
import math

def nll(names):
    total, count = 0.0, 0
    for w in names:
        s = "." + w + "."
        for a, b in zip(s, s[1:]):
            total -= math.log(P[stoi[a], stoi[b]].item())
            count += 1
    return total / count

print(round(nll(["emma"]), 3))                  # 2.514
print(round(nll(train), 3), round(nll(val), 3))  # 2.454 2.458
print(round(math.exp(nll(val)), 2))             # 11.68

## 9.3 The MLP language model and embeddings

Instead of counting pairs, a small neural network reads the last few letters. Each letter becomes a learned list of numbers, called its embedding. How many letters it can see, its context window, is fixed. That is the limit attention removes in Module 10.

[Read the lesson](https://aicourses.dev/research/language-models/mlp-language-model)

### The code

Start from the last lesson’s `train`, `val`, `chars` and `stoi`. First, turn names into examples: each example is three letters of context and the letter that follows.

In [ ]:
import math
import torch
import torch.nn.functional as F

block = 3     # the context window: how many letters it sees

def build(names):
    X, Y = [], []
    for w in names:
        context = [0] * block        # start with "..."
        for ch in w + ".":
            X.append(context)        # what the model sees
            Y.append(stoi[ch])       # the letter it should guess
            context = context[1:] + [stoi[ch]]
    return torch.tensor(X), torch.tensor(Y)

X, Y = build(["emma"])
for x, y in zip(X.tolist(), Y.tolist()):
    print("".join(chars[i] for i in x), "->", chars[y])
Xtr, Ytr = build(train)
Xva, Yva = build(val)
print(Xtr.shape, Xva.shape)

Next, the parameters and the forward pass. The line `C[X]` is where every letter becomes its embedding: indexing a table with a tensor of row numbers picks all those rows at once. `.view(-1, 30)` is `reshape` under another name; it lays each example’s three strips end to end, and −1 tells PyTorch to work out that size itself, here the batch size.

In [ ]:
torch.manual_seed(42)
C  = torch.randn(27, 10)            # 10 numbers per letter
W1 = torch.randn(30, 200) * 0.2     # 3 letters x 10 numbers in
b1 = torch.zeros(200)
W2 = torch.randn(200, 27) * 0.01    # small, so first guesses are even
b2 = torch.zeros(27)
params = [C, W1, b1, W2, b2]
for p in params:
    p.requires_grad = True
print(sum(p.numel() for p in params))          # 11897

def forward(X):
    emb = C[X]                       # (batch, 3, 10)
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)   # (batch, 200)
    return h @ W2 + b2               # (batch, 27) scores

print(round(F.cross_entropy(forward(Xva), Yva).item(), 3))   # 3.31

Now Module 8’s training loop, with a learning rate that drops for the last third:

In [ ]:
for step in range(30000):
    ix = torch.randint(0, len(Xtr), (32,))    # a batch of 32
    loss = F.cross_entropy(forward(Xtr[ix]), Ytr[ix])
    for p in params:
        p.grad = None
    loss.backward()
    lr = 0.1 if step < 20000 else 0.01        # decay near the end
    for p in params:
        p.data -= lr * p.grad
    if step % 10000 == 0:
        print(step, round(loss.item(), 3))

with torch.no_grad():
    tr = F.cross_entropy(forward(Xtr), Ytr).item()
    va = F.cross_entropy(forward(Xva), Yva).item()
print(round(tr, 3), round(va, 3))   # 2.121 2.142
print(round(math.exp(va), 2))   # perplexity: 8.52

Sampling works as in lesson 9.2, except the model now reads the last three letters each time:

In [ ]:
g = torch.Generator().manual_seed(42)
for _ in range(6):
    context, name = [0] * block, ""
    while True:
        with torch.no_grad():
            logits = forward(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        i = torch.multinomial(probs, 1, generator=g).item()
        if i == 0:
            break
        name += chars[i]
        context = context[1:] + [i]
    print(name)

## 9.4 Sampling: temperature, top-k, top-p

A model gives probabilities, and sampling picks the actual next letter. Temperature makes the picks bolder or safer, and top-k and top-p cut off unlikely choices. You use these settings every time you generate text later.

[Read the lesson](https://aicourses.dev/research/language-models/sampling)

### Temperature: bolder or safer

Work it through with three letters whose scores are 2, 1 and 0. With $T = 1$: $e^2 \approx 7.389$, $e^1 \approx 2.718$ and $e^0 = 1$, which add up to 11.107, so the probabilities are 0.665, 0.245 and 0.090. With $T = 0.5$, the scores become 4, 2 and 0, the gaps double, and the probabilities sharpen to 0.867, 0.117 and 0.016. With $T = 2$, the scores become 1, 0.5 and 0, and the probabilities flatten to 0.506, 0.307 and 0.186.

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.0])
for T in [0.5, 1.0, 2.0]:
    probs = F.softmax(logits / T, dim=0)
    print(T, [round(p, 3) for p in probs.tolist()])
# 0.5 [0.867, 0.117, 0.016]
# 1.0 [0.665, 0.245, 0.09]
# 2.0 [0.506, 0.307, 0.186]

### The code

One function does all three. It takes the model’s scores for the next letter and returns the index of the letter it picked.

In [ ]:
def pick(logits, temperature=1.0, top_k=None, top_p=None,
         generator=None):
    logits = logits / temperature
    if top_k is not None:              # keep only the k best
        kth = torch.topk(logits, top_k).values[-1]
        logits[logits < kth] = -float("inf")
    probs = F.softmax(logits, dim=-1)
    if top_p is not None:              # keep the smallest set
        p, idx = torch.sort(probs, descending=True)
        before = torch.cumsum(p, dim=-1) - p   # mass ahead of it
        probs[idx[before >= top_p]] = 0        # already enough
        probs = probs / probs.sum()
    return torch.multinomial(probs, 1, generator=generator).item()

Now wrap it in the name-writing loop, using `forward`, `block` and `chars` from lesson 9.3:

In [ ]:
def write(temperature=1.0, top_k=None, top_p=None, generator=None):
    context, name = [0] * block, ""
    while True:
        with torch.no_grad():
            logits = forward(torch.tensor([context]))[0]
        i = pick(logits, temperature, top_k, top_p, generator)
        if i == 0:                     # picked ".": the name ends
            return name
        name += chars[i]
        context = context[1:] + [i]

g = torch.Generator().manual_seed(42)
for t in [0.5, 1.0, 1.5]:
    print(t, [write(t, generator=g) for _ in range(5)])

Five names prove little, so count. This writes 1,000 names at each temperature and counts how many are copies of a training name, and the average length:

In [ ]:
seen = set(train)
g = torch.Generator().manual_seed(0)
for t in [0.5, 1.0, 1.5]:
    names = [write(t, generator=g) for _ in range(1000)]
    copies = sum(name in seen for name in names)
    print(t, copies, sum(map(len, names)) / 1000)
# 0.5 587 5.116
# 1.0 201 6.34
# 1.5 78 8.197

## 9.5 Entropy, cross-entropy and KL divergence

Entropy measures how unsure a model’s guesses are. KL divergence measures how far one model’s guesses have moved from another’s. Later, KL keeps a tuned model close to where it started, and falling entropy warns that a model has stopped trying new answers.

[Read the lesson](https://aicourses.dev/research/language-models/entropy-and-kl)

### The code

Each formula is one line in PyTorch:

In [ ]:
def entropy(p):
    return -(p * p.log()).sum().item()

def cross_entropy(p, q):
    return -(p * q.log()).sum().item()

def kl(p, q):
    return (p * (p / q).log()).sum().item()

truth = torch.tensor([0.5, 0.5])     # sun, rain
asha = torch.tensor([0.9, 0.1])      # Asha's forecast
print(round(entropy(truth), 3), round(cross_entropy(truth, asha), 3))
print(round(kl(truth, asha), 3), round(kl(asha, truth), 3))
# 0.693 1.204
# 0.511 0.368
print(F.kl_div(asha.log(), truth, reduction="sum").item())
# 0.5108256340026855

Now measure your name model, with `forward` and `stoi` from lesson 9.3:

In [ ]:
for ctx in ["...", "emm", ".qu"]:
    x = torch.tensor([[stoi[c] for c in ctx]])
    with torch.no_grad():
        logits = forward(x)[0]
    for T in [0.5, 1.0, 1.5]:
        p = F.softmax(logits / T, dim=0)
        print(ctx, T, round(entropy(p), 2))

## 9.6 Manual backprop through the MLP language model

Deriving the softmax-cross-entropy gradient (p − one-hot) by hand can come up in interviews, but it is slow going, so it never blocks the next lesson.

[Read the lesson](https://aicourses.dev/research/language-models/manual-backprop)

### Deriving it

Check it by nudging, as Module 7 taught:

In [ ]:
z = torch.tensor([2.0, 1.0, 0.0])
y = 0                                    # the right letter
p = F.softmax(z, dim=0)
grad = p.clone()
grad[y] -= 1                             # p minus one-hot
print([round(g, 3) for g in grad.tolist()])   # [-0.335, 0.245, 0.09]

def loss_at(z):
    return F.cross_entropy(z.unsqueeze(0), torch.tensor([y]))

nudge = torch.tensor([1e-4, 0.0, 0.0])
slope = (loss_at(z + nudge) - loss_at(z - nudge)) / 2e-4
print(round(slope.item(), 3))            # -0.335

### Back through the whole network

First, a forward pass that keeps every step, and autograd’s answers to check against. `C`, `W1`, `b1`, `W2`, `b2` and `Xtr`, `Ytr` are from lesson 9.3:

In [ ]:
torch.manual_seed(42)
ix = torch.randint(0, len(Xtr), (32,))
X, Y = Xtr[ix], Ytr[ix]
emb = C[X].view(-1, 30)                  # (32, 30)
h = torch.tanh(emb @ W1 + b1)            # (32, 200)
logits = h @ W2 + b2                     # (32, 27)
loss = F.cross_entropy(logits, Y)
for t in [C, W1, b1, W2, b2]:
    t.grad = None
loss.backward()                          # autograd's answers

Now every gradient by hand, from the loss back to the embedding table:

In [ ]:
with torch.no_grad():
    dlogits = F.softmax(logits, dim=1)
    dlogits[range(32), Y] -= 1           # p minus one-hot
    dlogits /= 32                        # the loss is a mean
    dW2 = h.T @ dlogits                  # (200, 27)
    db2 = dlogits.sum(0)                 # (27,)
    dh = dlogits @ W2.T                  # (32, 200)
    da = dh * (1 - h**2)                 # through tanh
    dW1 = emb.T @ da                     # (30, 200)
    db1 = da.sum(0)                      # (200,)
    demb = (da @ W1.T).view(-1, 3, 10)   # (32, 3, 10)
    dC = torch.zeros_like(C)
    dC.index_add_(0, X.view(-1), demb.view(-1, 10))

for name, mine, auto in [("C", dC, C), ("W1", dW1, W1),
        ("b1", db1, b1), ("W2", dW2, W2), ("b2", db2, b2)]:
    diff = (mine - auto.grad).abs().max().item()
    print(name, f"{diff:.1e}")